# LeetCode 3272 — Find the Count of Good Integers bằng Python

- Link: https://leetcode.com/problems/find-the-count-of-good-integers/
- Độ khó: **Hard**
- Chủ đề: **Liệt kê palindrome** qua nửa đầu (`≤ 10^5`), **khử trùng theo đa tập chữ số**, **đếm hoán vị có lặp** không có số 0 đứng đầu

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countGoodIntegers(self, n: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Số "tốt" chỉ phụ thuộc **đa tập chữ số** của nó: tốt ⇔ đa tập đó có một cách xếp thành palindrome (không số 0 đầu) chia hết cho `k`.
2. Thay vì duyệt `9·10^{n−1}` số, duyệt mọi **palindrome** `n` chữ số (xác định bởi nửa đầu ⇒ `≤ 9·10^4` cái), lọc chia hết cho `k`, thu tập các đa tập (dạng chuỗi chữ số đã sắp).
3. Với mỗi đa tập hợp lệ, đếm số số `n` chữ số tạo từ nó (không số 0 đầu): `n!/∏c_d! − [c_0 > 0]·(n−1)!/((c_0−1)!∏_{d≠0} c_d!)`.
4. Khử trùng là bắt buộc — nhiều palindrome chung một đa tập.
5. Kiểm chứng bằng vét cạn với `n` nhỏ.

## 2. Đề bài & ví dụ

Số `x` là **k-palindromic** nếu `x` là palindrome và chia hết cho `k`. Số **tốt** nếu các chữ số của nó có thể sắp lại thành một số k-palindromic. Đếm số tốt có đúng `n` chữ số. (Không số nào được có số 0 đứng đầu — cả trước lẫn sau khi sắp.)

**Ví dụ 1:** `n = 3, k = 5` → `27`. **Ví dụ 2:** `n = 1, k = 4` → `2` (`4, 8`). **Ví dụ 3:** `n = 5, k = 6` → `2468`.

**Ràng buộc:** `1 ≤ n ≤ 10`; `1 ≤ k ≤ 9`.

In [1]:
from typing import List
from math import factorial
from collections import Counter
from itertools import permutations

examples = [(3, 5, 27), (1, 4, 2), (5, 6, 2468)]
for n, k, exp in examples:
    print(f"n = {n}, k = {k} -> kỳ vọng {exp}")

n = 3, k = 5 -> kỳ vọng 27
n = 1, k = 4 -> kỳ vọng 2
n = 5, k = 6 -> kỳ vọng 2468


## 3. Kiến thức nền

### 3.1. Dựng palindrome từ nửa đầu

Palindrome `n` chữ số xác định bởi `⌈n/2⌉` chữ số đầu (chữ số đầu khác 0): ghép nửa đầu với **đảo ngược** của nửa đầu bỏ (hoặc không bỏ) chữ số giữa.

In [2]:
def palindromes(n):
    h = (n + 1) // 2
    for left in range(10 ** (h - 1), 10 ** h):
        s = str(left)
        yield s + s[::-1][n % 2:]

print(list(palindromes(3))[:5], '...', sum(1 for _ in palindromes(3)), 'palindrome 3 chữ số')

['101', '111', '121', '131', '141'] ... 90 palindrome 3 chữ số


### 3.2. Đếm số `n` chữ số từ một đa tập (không số 0 đầu)

Tổng số cách xếp `n!/∏ c_d!`; trừ các cách có `0` đứng đầu: cố định `0` ở đầu, xếp `n − 1` chữ số còn lại.

In [3]:
def arrangements_no_leading_zero(cnt):
    n = sum(cnt)
    total = factorial(n)
    for c in cnt:
        total //= factorial(c)
    if cnt[0] > 0:
        bad = factorial(n - 1) // factorial(cnt[0] - 1)
        for d in range(1, 10):
            bad //= factorial(cnt[d])
        total -= bad
    return total

cnt = [0] * 10
for ch in "505":
    cnt[int(ch)] += 1
print('đa tập {0, 5, 5}:', arrangements_no_leading_zero(cnt), '(505, 550)')

đa tập {0, 5, 5}: 2 (505, 550)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`n = 3`, `k = 5`): palindrome 3 chữ số chia hết cho 5 phải tận cùng bằng `5` (tận cùng `0` thì chữ số đầu cũng `0`) ⇒ dạng `5x5`, `x = 0..9` ⇒ 10 palindrome, 10 đa tập `{5, 5, x}`.

- `x = 0`: đa tập `{0, 5, 5}` ⇒ 2 số (`505`, `550`).
- `x = 5`: `{5, 5, 5}` ⇒ 1 số.
- 8 giá trị `x` khác: mỗi đa tập `{5, 5, x}` ⇒ 3 số.

Tổng `2 + 1 + 8·3 = 27` ✓.

## 5. Cách 1 — Brute Force: duyệt mọi số `n` chữ số

In [4]:
def brute(n, k):
    good_sets = set()
    for p in palindromes(n):
        if int(p) % k == 0:
            good_sets.add(''.join(sorted(p)))
    return sum(1 for x in range(10 ** (n - 1), 10 ** n) if ''.join(sorted(str(x))) in good_sets)

def brute_pure(n, k):
    """Không dùng palindromes(): thử mọi hoán vị của mọi số (rất chậm, n <= 4)."""
    cnt = 0
    for x in range(10 ** (n - 1), 10 ** n):
        s = str(x)
        if any(p[0] != '0' and p == p[::-1] and int(''.join(p)) % k == 0 for p in set(permutations(s))):
            cnt += 1
    return cnt

for n, k, exp in examples:
    print(brute(n, k), '| kỳ vọng', exp)
print('vét cạn thuần (n = 3, k = 5):', brute_pure(3, 5))

27 | kỳ vọng 27
2 | kỳ vọng 2
2468 | kỳ vọng 2468
vét cạn thuần (n = 3, k = 5): 27


**Độ phức tạp:** `O(9·10^{n−1} · n log n)` — tới `9·10^9` khi `n = 10`.

## 6. Ý tưởng thuật toán / bước đột phá

```
seen = set()
for p in palindromes(n):                       # ≤ 9·10^4
    if int(p) % k == 0: seen.add(sorted digits of p)
answer = Σ_{s ∈ seen} arrangements_no_leading_zero(cnt(s))
```

Tổng chi phí `O(10^{⌈n/2⌉} · n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Liệt kê palindrome + khử trùng đa tập (lời giải chính)

In [5]:
def palindrome_solution(n, k):
    seen = set()
    for p in palindromes(n):
        if int(p) % k == 0:
            seen.add(''.join(sorted(p)))
    total = 0
    for s in seen:
        cnt = [0] * 10
        for ch in s:
            cnt[ord(ch) - 48] += 1
        total += arrangements_no_leading_zero(cnt)
    return total

for n, k, exp in examples:
    print(palindrome_solution(n, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
2 | kỳ vọng 2
2468 | kỳ vọng 2468


### 7.2. Cách 3 — Liệt kê đa tập, kiểm tra có palindrome chia hết cho `k`

Đa tập có thể xếp thành palindrome ⇔ tối đa một chữ số có số lần lẻ (và đúng một nếu `n` lẻ). Với mỗi đa tập như vậy, thử các hoán vị **phân biệt** của nửa đầu (không bắt đầu bằng 0). Duyệt đa tập: `C(n + 9, 9)` cái (`92 378` với `n = 10`).

In [6]:
from itertools import combinations_with_replacement

def multiset_solution(n, k):
    total = 0
    for combo in combinations_with_replacement(range(10), n):
        cnt = [0] * 10
        for d in combo:
            cnt[d] += 1
        odd = [d for d in range(10) if cnt[d] % 2]
        if len(odd) > 1:
            continue
        half = ''.join(str(d) * (cnt[d] // 2) for d in range(10))
        mid = str(odd[0]) if odd else ''
        ok = False
        for perm in set(permutations(half)):
            if perm and perm[0] == '0':
                continue
            left = ''.join(perm)
            s = left + mid + left[::-1]
            if s[0] != '0' and int(s) % k == 0:
                ok = True
                break
        if ok:
            total += arrangements_no_leading_zero(cnt)
    return total

for n, k, exp in examples:
    print(multiset_solution(n, k), '| kỳ vọng', exp)

27 | kỳ vọng 27
2 | kỳ vọng 2
2468 | kỳ vọng 2468


Với `n = 1`, `half = ''`, palindrome là chữ số giữa — xử lý tự nhiên (điều kiện `s[0] != '0'` loại số `0`).

### 7.3. Cách 4 — Bảng tra trước cho mọi `(n, k)`

Vì chỉ có `10 · 9 = 90` cặp `(n, k)`, có thể tính sẵn toàn bộ bảng (vài giây) rồi trả lời `O(1)`. Ở đây ta tính bảng cho `n ≤ 6` để minh họa.

In [7]:
TABLE = {(n, k): palindrome_solution(n, k) for n in range(1, 7) for k in range(1, 10)}
def table_lookup(n, k):
    return TABLE[(n, k)] if (n, k) in TABLE else palindrome_solution(n, k)

print('bảng n = 3:', [TABLE[(3, k)] for k in range(1, 10)])

bảng n = 3: [243, 108, 69, 54, 27, 30, 33, 27, 23]


## 8. TRACE từng bước

Trace cách 2 với `n = 3`, `k = 5`.

In [8]:
def trace(n, k):
    seen = {}
    for p in palindromes(n):
        if int(p) % k == 0:
            seen.setdefault(''.join(sorted(p)), []).append(p)
    total = 0
    for s, ps in sorted(seen.items()):
        cnt = [0] * 10
        for ch in s:
            cnt[int(ch)] += 1
        a = arrangements_no_leading_zero(cnt)
        total += a
        print(f"đa tập {s}: từ palindrome {ps} -> {a} số")
    print('tổng:', total, '| vét cạn:', brute(n, k))

trace(3, 5)

đa tập 055: từ palindrome ['505'] -> 2 số
đa tập 155: từ palindrome ['515'] -> 3 số
đa tập 255: từ palindrome ['525'] -> 3 số
đa tập 355: từ palindrome ['535'] -> 3 số
đa tập 455: từ palindrome ['545'] -> 3 số
đa tập 555: từ palindrome ['555'] -> 1 số
đa tập 556: từ palindrome ['565'] -> 3 số
đa tập 557: từ palindrome ['575'] -> 3 số
đa tập 558: từ palindrome ['585'] -> 3 số
đa tập 559: từ palindrome ['595'] -> 3 số
tổng: 27 | vét cạn: 27


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countGoodIntegers(self, n: int, k: int) -> int:
        fact = [1] * (n + 1)
        for i in range(1, n + 1):
            fact[i] = fact[i - 1] * i
        h = (n + 1) // 2
        cut = n % 2
        seen = set()
        for left in range(10 ** (h - 1), 10 ** h):
            s = str(left)
            p = s + s[::-1][cut:]
            if int(p) % k == 0:
                seen.add(''.join(sorted(p)))
        total = 0
        for s in seen:
            cnt = [0] * 10
            for ch in s:
                cnt[ord(ch) - 48] += 1
            ways = fact[n]
            for c in cnt:
                ways //= fact[c]
            if cnt[0]:
                bad = fact[n - 1] // fact[cnt[0] - 1]
                for d in range(1, 10):
                    bad //= fact[cnt[d]]
                ways -= bad
            total += ways
        return total

for n, k, exp in examples:
    got = Solution().countGoodIntegers(n, k)
    print(got, 'OK' if got == exp else 'SAI')

27 OK
2 OK
2468 OK


## 10. Vì sao thuật toán đúng

1. **Tính tốt phụ thuộc đa tập:** hai số có cùng đa tập chữ số có cùng tập các cách sắp xếp lại ⇒ cùng tốt hoặc cùng không tốt.
2. **Đa tập tốt ⇔ chứa một palindrome hợp lệ:** đa tập `M` tốt ⇔ tồn tại palindrome `n` chữ số (không số 0 đầu) có đa tập `M` và chia hết cho `k`. Duyệt mọi palindrome như vậy và gom đa tập cho đúng tập `M` tốt (không trùng nhờ `set`).
3. **Đếm số có đa tập `M`:** số chuỗi `n` ký tự với đa tập `M` là hệ số đa thức; trừ những chuỗi bắt đầu bằng `0` (đếm bằng cố định `0` đầu). Chuỗi còn lại đúng là các số `n` chữ số.
4. Các đa tập tốt khác nhau cho các tập số rời nhau ⇒ cộng lại.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt mọi số | `O(10^n · n)` | `O(10^{n/2})` |
| 2. Palindrome + đa tập | `O(10^{⌈n/2⌉} · n)` | `O(10^{⌈n/2⌉})` |
| 3. Đa tập + thử nửa đầu | `O(C(n+9, 9) · (n/2)!)` (xấu) | `O(1)` |
| 4. Bảng tra | `O(1)` sau tiền xử lý | `O(90)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'palindrome': palindrome_solution, 'multiset': multiset_solution, 'table': table_lookup,
         'submit': lambda n, k: Solution().countGoodIntegers(n, k)}
edge = [(1, 1, 9), (1, 9, 1), (2, 1, 9), (2, 2, 4), (2, 3, 3), (4, 1, None)]
for n, k, exp in examples + edge:
    e = brute(n, k)
    if exp is not None:
        assert e == exp, (n, k, e)
    for name, f in impls.items():
        assert f(n, k) == e, (name, n, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu đầy đủ với vét cạn cho `n ≤ 5`, mọi `k`

In [11]:
cnt = 0
for n in range(1, 6):
    for k in range(1, 10):
        exp = brute(n, k)
        for name, f in impls.items():
            if name == 'multiset' and n > 5:
                continue
            assert f(n, k) == exp, (name, n, k)
        cnt += 1
for n in range(1, 4):
    for k in range(1, 10):
        assert brute_pure(n, k) == brute(n, k)
print('Khớp vét cạn trên', cnt, 'cặp (n, k); vét cạn thuần (thử mọi hoán vị) khớp với n <= 3.')

Khớp vét cạn trên 45 cặp (n, k); vét cạn thuần (thử mọi hoán vị) khớp với n <= 3.


### 12.3. Ca lớn `n = 10`, mọi `k`

In [12]:
import time
t = time.time()
res10 = [Solution().countGoodIntegers(10, k) for k in range(1, 10)]
t1 = time.time() - t
print(f'n = 10, k = 1..9: {res10} | {t1:.2f}s')
t = time.time(); assert multiset_solution(8, 7) == Solution().countGoodIntegers(8, 7); print(f'cách 3 khớp với n = 8, k = 7 ({time.time() - t:.2f}s)')

n = 10, k = 1..9: [41457024, 39718144, 13831104, 37326452, 19284856, 13249798, 40242031, 35755906, 4610368] | 0.61s
cách 3 khớp với n = 8, k = 7 (0.03s)


Kiểm tra độc lập: với `k = 1`, số tốt ⇔ đa tập xếp được thành palindrome không số 0 đầu. Với `n` chẵn, `k = 1`, ta có thể đếm trực tiếp bằng duyệt đa tập (mọi chữ số xuất hiện chẵn lần, có ít nhất một chữ số khác 0).

In [13]:
def k1_direct(n):
    total = 0
    for combo in combinations_with_replacement(range(10), n):
        c = [0] * 10
        for d in combo:
            c[d] += 1
        odd = sum(x % 2 for x in c)
        if odd <= n % 2 and any(c[d] for d in range(1, 10)):
            # cần một chữ số khác 0 xuất hiện >= 2 lần để đứng hai đầu (trừ n = 1)
            if n == 1 or any(c[d] >= 2 for d in range(1, 10)):
                total += arrangements_no_leading_zero(c)
    return total

for n in [1, 2, 3, 6, 10]:
    assert k1_direct(n) == Solution().countGoodIntegers(n, 1), n
print('k = 1: khớp đếm trực tiếp theo đa tập với n = 1, 2, 3, 6, 10.')

k = 1: khớp đếm trực tiếp theo đa tập với n = 1, 2, 3, 6, 10.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm các palindrome chia hết cho `k`

In [14]:
def wrong_count_palindromes(n, k):
    return sum(1 for p in palindromes(n) if int(p) % k == 0)     # SAI

print('sai :', wrong_count_palindromes(3, 5), '| đúng:', Solution().countGoodIntegers(3, 5))

sai : 10 | đúng: 27


**Vì sao sai:** đề đếm mọi số **có thể sắp lại** thành palindrome như vậy (`551` được tính vì sắp thành `515`).

### Lỗi 2 — Không khử trùng đa tập

In [15]:
def wrong_no_dedup(n, k):
    total = 0
    for p in palindromes(n):
        if int(p) % k == 0:
            cnt = [0] * 10
            for ch in p:
                cnt[int(ch)] += 1
            total += arrangements_no_leading_zero(cnt)           # SAI: đa tập lặp lại bị cộng nhiều lần
    return total

print('sai :', wrong_no_dedup(5, 6), '| đúng:', Solution().countGoodIntegers(5, 6))

sai : 2996 | đúng: 2468


**Vì sao sai:** nhiều palindrome có **cùng** đa tập chữ số (ví dụ `20802` và `80208` đều chia hết cho 6 và cùng đa tập `{0,0,2,2,8}`) — các số tốt tương ứng bị đếm nhiều lần.

### Lỗi 3 — Quên trừ các cách xếp có số 0 đứng đầu

In [16]:
def wrong_leading_zero(n, k):
    seen = {''.join(sorted(p)) for p in palindromes(n) if int(p) % k == 0}
    total = 0
    for s in seen:
        c = Counter(s); w = factorial(n)
        for v in c.values():
            w //= factorial(v)
        total += w                                                # SAI
    return total

print('sai :', wrong_leading_zero(3, 5), '| đúng:', Solution().countGoodIntegers(3, 5))

sai : 28 | đúng: 27


**Vì sao sai:** đa tập `{0, 5, 5}` có 3 chuỗi `055, 505, 550` nhưng `055` không phải số 3 chữ số.

### Lỗi 4 — Cho phép palindrome có số 0 đứng đầu (duyệt nửa đầu từ 0)

In [17]:
def wrong_leading_zero_palindromes(n, k):
    h = (n + 1) // 2
    seen = set()
    for left in range(0, 10 ** h):                                # SAI: nửa đầu có thể bắt đầu bằng 0
        s = str(left).zfill(h)
        p = s + s[::-1][n % 2:]
        if int(p) % k == 0:
            seen.add(''.join(sorted(p)))
    total = 0
    for s in seen:
        cnt = [0] * 10
        for ch in s:
            cnt[int(ch)] += 1
        total += arrangements_no_leading_zero(cnt)
    return total

print('sai :', wrong_leading_zero_palindromes(3, 5), '| đúng:', Solution().countGoodIntegers(3, 5))

sai : 36 | đúng: 27


**Vì sao sai:** "palindrome" `0x0` không phải số 3 chữ số; các đa tập `{0, 0, x}` không xếp được thành palindrome hợp lệ (`x00` không phải palindrome) nhưng bị tính — thêm 9 số `100, 200, …, 900`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có thể sắp lại chữ số để …" ⇒ làm việc với **đa tập chữ số**.
- Palindrome `n` chữ số ⇔ nửa đầu `⌈n/2⌉` chữ số — giảm không gian từ `10^n` xuống `10^{n/2}`.
- Đếm hoán vị có lặp với điều kiện chữ số đầu ⇒ tổng trừ phần "0 đứng đầu".

## 15. Biến thể của bài

1. Đếm số tốt **≤ N** cho trước — kết hợp với digit DP trên đa tập (khó hơn).
2. Điều kiện "chia hết cho `k`" thay bằng "tổng chữ số chia hết cho `k`" ⇒ chỉ phụ thuộc đa tập, không cần thử palindrome.
3. Liệt kê luôn các số tốt nhỏ nhất của mỗi đa tập.
4. Cơ số `b` khác 10.

In [18]:
# Biến thể 2: tổng chữ số chia hết cho k (chỉ phụ thuộc đa tập)
def digit_sum_variant(n, k):
    total = 0
    for combo in combinations_with_replacement(range(10), n):
        c = [0] * 10
        for d in combo:
            c[d] += 1
        if sum(combo) % k:
            continue
        if sum(x % 2 for x in c) > n % 2:
            continue
        if n > 1 and not any(c[d] >= 2 for d in range(1, 10)):
            continue
        if n == 1 and c[0]:
            continue
        total += arrangements_no_leading_zero(c)
    return total

def digit_sum_brute(n, k):
    ok = {''.join(sorted(p)) for p in palindromes(n) if sum(map(int, p)) % k == 0}
    return sum(1 for x in range(10 ** (n - 1), 10 ** n) if ''.join(sorted(str(x))) in ok)

for n in range(1, 5):
    for k in range(1, 10):
        assert digit_sum_variant(n, k) == digit_sum_brute(n, k), (n, k)
print('Biến thể "tổng chữ số chia hết cho k" khớp vét cạn với n <= 4.')

Biến thể "tổng chữ số chia hết cho k" khớp vét cạn với n <= 4.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Duyệt mọi số | `O(10^n)` | Oracle |
| 2 | Palindrome + đa tập | `O(10^{n/2} n)` | **Nộp bài** |
| 3 | Đa tập + thử nửa đầu | phụ thuộc đa tập | Góc nhìn ngược |
| 4 | Bảng tra | `O(1)` | Chỉ 90 đầu vào |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2`, `k = 9`.

**Bài 2 (tính tay):** `n = 2`, `k = 4`.

**Bài 3 (điền TODO):** liệt kê các đa tập tốt (dạng chuỗi đã sắp) cho `(n, k)`.

**Bài 4 (tự viết lại):** với mỗi đa tập tốt, trả về **số nhỏ nhất** tạo được.

**Bài 5 (mini challenge):** biến thể 4 — cơ số `b` (`2 ≤ b ≤ 10`).

In [19]:
def good_multisets_todo(n, k):
    # TODO
    raise NotImplementedError

def smallest_per_multiset_todo(n, k):
    # TODO
    raise NotImplementedError

def base_b_todo(n, k, b):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Palindrome 2 chữ số: `11, 22, …, 99`. (2) Chia hết cho 9: chỉ `99`. (3) `1`.

**Bài 2.** (1) `44`, `88`. (2) Mỗi đa tập một số. (3) `2`.

**Bài 3.** (1) Tập `seen` của cách 2. (2) Sắp. (3) Trả danh sách.

**Bài 4.** (1) Chữ số khác 0 nhỏ nhất đứng đầu. (2) Phần còn lại sắp tăng. (3) Kiểm có đúng đa tập.

**Bài 5.** (1) Palindrome trong cơ số `b` từ nửa đầu. (2) Giá trị `int(p, b) % k`. (3) Đếm hoán vị như cũ (chữ số `0..b−1`).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3, 4
assert Solution().countGoodIntegers(2, 9) == 1
assert Solution().countGoodIntegers(2, 4) == 2

def good_multisets(n, k):
    return sorted({''.join(sorted(p)) for p in palindromes(n) if int(p) % k == 0})

def smallest_per_multiset(n, k):
    out = []
    for s in good_multisets(n, k):
        digits = sorted(s)
        first = next(d for d in digits if d != '0')
        digits.remove(first)
        out.append(first + ''.join(digits))
    return out

ms = good_multisets(3, 5)
sm = smallest_per_multiset(3, 5)
for s, x in zip(ms, sm):
    assert ''.join(sorted(x)) == s and x[0] != '0'
    assert x == min(''.join(p) for p in set(permutations(s)) if p[0] != '0')
print('Bài 1: 1 | Bài 2: 2 | Bài 3:', ms[:4], '... | Bài 4:', sm[:4], '...')

Bài 1: 1 | Bài 2: 2 | Bài 3: ['055', '155', '255', '355'] ... | Bài 4: ['505', '155', '255', '355'] ...


In [21]:
# Bài 5
def to_base(x, b):
    out = []
    while x:
        out.append(str(x % b)); x //= b
    return ''.join(reversed(out)) or '0'

def base_b(n, k, b):
    h = (n + 1) // 2
    seen = set()
    for left in range(b ** (h - 1), b ** h):
        s = to_base(left, b)
        p = s + s[::-1][n % 2:]
        if int(p, b) % k == 0:
            seen.add(''.join(sorted(p)))
    total = 0
    for s in seen:
        c = [0] * b
        for ch in s:
            c[int(ch)] += 1
        w = factorial(n)
        for v in c:
            w //= factorial(v)
        if c[0]:
            bad = factorial(n - 1) // factorial(c[0] - 1)
            for d in range(1, b):
                bad //= factorial(c[d])
            w -= bad
        total += w
    return total

def base_b_brute(n, k, b):
    ok = set()
    h = (n + 1) // 2
    for left in range(b ** (h - 1), b ** h):
        s = to_base(left, b); p = s + s[::-1][n % 2:]
        if int(p, b) % k == 0:
            ok.add(''.join(sorted(p)))
    return sum(1 for x in range(b ** (n - 1), b ** n) if ''.join(sorted(to_base(x, b))) in ok)

for b in range(2, 11):
    for n in range(1, 5):
        for k in range(1, 10):
            assert base_b(n, k, b) == base_b_brute(n, k, b)
assert all(base_b(n, k, 10) == Solution().countGoodIntegers(n, k) for n in range(1, 6) for k in range(1, 10))
print('Bài 5: cơ số b khớp vét cạn; với b = 10 trùng bài gốc.')

Bài 5: cơ số b khớp vét cạn; với b = 10 trùng bài gốc.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
seen = {sorted(p) for p in palindromes(n) (nửa đầu không bắt đầu bằng 0) if int(p) % k == 0}
ans = Σ_{s ∈ seen} n!/∏c! − [c0>0]·(n−1)!/((c0−1)!·∏_{d≠0} c_d!)
```

**Checklist:**

- [ ] Duyệt nửa đầu từ `10^{h−1}` (không số 0 đầu)?
- [ ] Khử trùng theo đa tập?
- [ ] Trừ các cách xếp có số 0 đứng đầu?
- [ ] Ghép palindrome đúng với `n` lẻ (bỏ chữ số giữa khi đảo)?
- [ ] Đếm số **có thể sắp lại**, không chỉ palindrome?